In [ ]:
# ------------------------------------------------------------------------------
# Cell 1: Install & Upgrade Dependencies
# Note: In Colab, if you run this for the first time, click 'Runtime -> Restart Session'
# ------------------------------------------------------------------------------
!pip install -q -U bitsandbytes transformers torch accelerate

In [ ]:
# Cell 2: Setup Device & Inspect Deep Neural Network Architecture
# ------------------------------------------------------------------------------
import gc
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextStreamer
)

# Check CUDA availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on device: {device}")

print("\n=== 1. Inspecting Deep Neural Network Architecture ===")
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

print(f"Loaded Model Class: {type(model).__name__}")
print("\n--- Structural Layer Layout ---")
print(model)

In [ ]:
# ------------------------------------------------------------------------------
# Cell 3: Direct Tensor Inference & Generation
# ------------------------------------------------------------------------------
print("\n=== 2. Direct Tensor Inference & Generation ===")
messages = [
    {"role": "system", "content": "You are a witty data science assistant."},
    {"role": "user", "content": "Tell a lighthearted joke for a room of data scientists."}
]

# Apply chat template and extract input tensors directly
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(device)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print(f"Input Tensor Shape (Batch, Seq Length): {input_ids.shape}")

# Execute direct model generate call
with torch.no_grad():
    output_tokens = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id
    )

# Slice away prompt tokens to isolate generated tokens
new_tokens = output_tokens[0][input_ids.shape[1]:]
decoded_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("\n--- Model Output ---")
print(decoded_output.strip())

# Clean up unquantized model memory before quantization
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# Cell 4: Native FP16 Model Loading & Real-Time Streaming
# ==============================================================================
print("\n=== 3. Native FP16 Streaming Inference ===")

# Load model directly in half-precision (torch.float16) to bypass bitsandbytes
streaming_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

quant_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

prompt_messages = [
    {"role": "user", "content": "Why did the neural network cross the road?"}
]

formatted_prompt = quant_tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs_quant = quant_tokenizer(formatted_prompt, return_tensors="pt").to(device)

# Instantiate streaming listener
streamer = TextStreamer(quant_tokenizer, skip_prompt=True, skip_special_tokens=True)

print("\n--- Real-Time Streaming Output ---")
with torch.no_grad():
    _ = streaming_model.generate(
        **inputs_quant,
        max_new_tokens=80,
        streamer=streamer,
        do_sample=True,
        temperature=0.8,
        pad_token_id=quant_tokenizer.eos_token_id
    )